In [0]:
from pyspark.sql.functions import *
from pyspark.sql.window import Window
from pyspark.sql.functions import max as spark_max, count as spark_count
from delta.tables import DeltaTable

In [0]:
SILVER = 'abfss://silver@logisticdatalakestorage.dfs.core.windows.net/'
GOLD   = 'abfss://gold@logisticdatalakestorage.dfs.core.windows.net/'

### Load the data

In [0]:
df_owm     = spark.read.format('delta').load(SILVER + 'openweathermap/')
df_tomtom  = spark.read.format('delta').load(SILVER + 'tomtom/')
df_meteo   = spark.read.format('delta').load(SILVER + 'openmeteo/')
df_waqi    = spark.read.format('delta').load(SILVER + 'waqi/')
df_rc      = spark.read.format('delta').load(SILVER + 'restcountries/')

In [0]:
df_owm.limit(5).display()
df_tomtom.limit(5).display()
df_meteo.limit(5).display()
df_rc.limit(5).display()
df_waqi.limit(5).display()

latitude,longitude,temperature,feels_like,humidity,pressure,temp_max,temp_min,wind_speed,wind_direction,rain_1h,visibility,event_time,year,month,day,hour,weather_main,weather_description,weather_code,city,country_code,join_time
13.7563,100.5018,31.76,38.76,73,1006,32.95,31.07,5.46,192,0.0,10000,2026-05-27T12:18:23.000Z,2026,5,27,12,Clouds,broken clouds,803,Bangkok,TH,2026-05-27T12:00:00.000Z
52.52,13.405,22.52,21.74,35,1023,23.32,21.19,5.14,330,0.0,10000,2026-05-27T12:16:29.000Z,2026,5,27,12,Clear,clear sky,800,Berlin,DE,2026-05-27T12:00:00.000Z
25.2048,55.2708,37.01,41.64,41,1005,38.18,37.01,5.14,270,0.0,10000,2026-05-27T12:21:00.000Z,2026,5,27,12,Clear,clear sky,800,Dubai,AE,2026-05-27T12:00:00.000Z
51.5072,-0.1276,26.13,26.13,50,1026,27.96,23.97,3.13,74,0.0,10000,2026-05-27T12:14:54.000Z,2026,5,27,12,Clear,clear sky,800,London,GB,2026-05-27T12:00:00.000Z
19.076,72.8777,33.0,40.0,66,1008,33.0,32.95,5.66,260,0.0,7000,2026-05-27T12:13:01.000Z,2026,5,27,12,Haze,haze,721,Mumbai,IN,2026-05-27T12:00:00.000Z


year,month,day,hour,incident_type,icon_category,geometry_type,lon,lat,event_time,city,country_code,congestion_level,join_time
2026,5,27,12,Feature,7,LineString,151.221199476,-33.9641851085,2026-05-27T12:00:00.000Z,Sydney,AU,10,2026-05-27T12:00:00.000Z
2026,5,27,12,Feature,7,LineString,151.157910072,-33.95791675,2026-05-27T12:00:00.000Z,Sydney,AU,10,2026-05-27T12:00:00.000Z
2026,5,27,12,Feature,8,LineString,151.204196953,-33.9215138137,2026-05-27T12:00:00.000Z,Sydney,AU,7,2026-05-27T12:00:00.000Z
2026,5,27,12,Feature,8,LineString,151.2066981129,-33.9208875538,2026-05-27T12:00:00.000Z,Sydney,AU,7,2026-05-27T12:00:00.000Z
2026,5,27,12,Feature,6,LineString,151.241757267,-33.92006139,2026-05-27T12:00:00.000Z,Sydney,AU,8,2026-05-27T12:00:00.000Z


elevation,generationtime_ms,latitude,longitude,year,month,day,hour,temperature,rain,precipitation_probability,visibility,wind_speed_10m,weather_code,city,country_code,event_time,join_time
86.0,0.1970529556274414,-33.848858,151.19551,2026,5,27,12,17.4,0.0,0,640.0,5.4,3,Sydney,AU,2026-05-27T00:00:00.000Z,2026-05-27T00:00:00.000Z
86.0,0.1970529556274414,-33.848858,151.19551,2026,5,27,12,19.0,0.0,0,3620.0,4.9,3,Sydney,AU,2026-05-27T01:00:00.000Z,2026-05-27T01:00:00.000Z
86.0,0.1970529556274414,-33.848858,151.19551,2026,5,27,12,20.1,0.0,2,12280.0,2.4,2,Sydney,AU,2026-05-27T02:00:00.000Z,2026-05-27T02:00:00.000Z
86.0,0.1970529556274414,-33.848858,151.19551,2026,5,27,12,20.8,0.0,8,13340.0,2.6,1,Sydney,AU,2026-05-27T03:00:00.000Z,2026-05-27T03:00:00.000Z
86.0,0.1970529556274414,-33.848858,151.19551,2026,5,27,12,20.9,0.0,23,12260.0,4.8,1,Sydney,AU,2026-05-27T04:00:00.000Z,2026-05-27T04:00:00.000Z


country_name,official_name,region,timezone,capital,year,month,day,country_code,ingestion_date
India,Republic of India,Asia,UTC+05:30,New Delhi,2026,5,27,IN,2026-05-27
Singapore,Republic of Singapore,Asia,UTC+08:00,Singapore,2026,5,27,SG,2026-05-27
United Kingdom,United Kingdom of Great Britain and Northern Ireland,Europe,UTC-08:00,London,2026,5,27,GB,2026-05-27
United States,United States of America,Americas,UTC-12:00,"Washington, D.C.",2026,5,27,US,2026-05-27
France,French Republic,Europe,UTC-10:00,Paris,2026,5,27,FR,2026-05-27


latitude,longitude,dominant_pollutant,pm25,pm10,o3,no2,co,temperature,humidity,measurement_time,year,month,day,join_time,ingestion_date,city,country_code
-33.872468,151.213337,pm25,43,23,1.9,21.8,2.2,19.4,90.0,2026-05-27T07:00:00.000Z,2026,5,27,2026-05-27T00:00:00.000Z,2026-05-27,Sydney,AU
-23.544845659,-46.627675592,pm25,109,51,4.1,17.4,11.8,17.7,97.0,2026-05-27T09:00:00.000Z,2026,5,27,2026-05-27T00:00:00.000Z,2026-05-27,Sao Paulo,BR
1.3666667,103.8,pm25,68,39,32.0,null,4.0,33.0,52.0,2026-05-27T09:00:00.000Z,2026,5,27,2026-05-27T00:00:00.000Z,2026-05-27,Singapore,SG
13.7563309,100.5017651,pm25,39,22,22.5,4.7,0.1,37.0,52.0,2026-05-27T09:00:00.000Z,2026,5,27,2026-05-27T00:00:00.000Z,2026-05-27,Bangkok,TH
19.0863,72.8888,pm25,151,79,11.7,2.4,2.6,35.0,73.03,2026-05-27T06:30:00.000Z,2026,5,27,2026-05-27T00:00:00.000Z,2026-05-27,Mumbai,IN


In [0]:
print('OWM:         ', df_owm.count())
print('TomTom:      ', df_tomtom.count())
print('OpenMeteo:   ', df_meteo.count())
print('WAQI:        ', df_waqi.count())
print('RestCountries:', df_rc.count())

OWM:          10
TomTom:       3236
OpenMeteo:    1680
WAQI:         10
RestCountries: 10


### Prepare dimension tables

#### _DIM 1: dim_country — from REST Countries_

In [0]:
dim_country = df_rc.select(
    col('country_code'),
    col('country_name'),
    col('official_name'),
    col('region'),
    col('timezone'),
    col('capital')
).dropDuplicates(['country_code'])

In [0]:
dim_country.display()

country_code,country_name,official_name,region,timezone,capital
AE,United Arab Emirates,United Arab Emirates,Asia,UTC+04:00,Abu Dhabi
AU,Australia,Commonwealth of Australia,Oceania,UTC+05:00,Canberra
BR,Brazil,Federative Republic of Brazil,Americas,UTC-05:00,Brasília
DE,Germany,Federal Republic of Germany,Europe,UTC+01:00,Berlin
FR,France,French Republic,Europe,UTC-10:00,Paris
GB,United Kingdom,United Kingdom of Great Britain and Northern Ireland,Europe,UTC-08:00,London
IN,India,Republic of India,Asia,UTC+05:30,New Delhi
SG,Singapore,Republic of Singapore,Asia,UTC+08:00,Singapore
TH,Thailand,Kingdom of Thailand,Asia,UTC+07:00,Bangkok
US,United States,United States of America,Americas,UTC-12:00,"Washington, D.C."


### _DIM 2: dim_air_quality — from WAQI_

In [0]:
dim_air_quality = df_waqi.select(
    col('city'),
    col('country_code'),
    col('dominant_pollutant'),
    col('pm25'),
    col('pm10'),
    col('o3'),
    col('no2'),
    col('co'),
    col('ingestion_date')
).dropDuplicates(['city'])

In [0]:
dim_air_quality.display()

city,country_code,dominant_pollutant,pm25,pm10,o3,no2,co,ingestion_date
Bangkok,TH,pm25,39,22,22.5,4.7,0.1,2026-05-27
Berlin,DE,o3,21,11,28.1,2.8,0.1,2026-05-27
Dubai,AE,pm25,95,68,36.2,3.7,null,2026-05-27
London,GB,pm25,65,41,43.9,26.1,1.8,2026-05-27
Mumbai,IN,pm25,151,79,11.7,2.4,2.6,2026-05-27
New York,US,pm25,30,null,null,null,null,2026-05-27
Paris,FR,pm25,72,25,34.4,45.8,0.1,2026-05-27
Sao Paulo,BR,pm25,109,51,4.1,17.4,11.8,2026-05-27
Singapore,SG,pm25,68,39,32.0,null,4.0,2026-05-27
Sydney,AU,pm25,43,23,1.9,21.8,2.2,2026-05-27


### Prepare the FACT Table

In [0]:
df_owm_clean = df_owm.select(
    col('city'),
    col('country_code'),
    col('join_time'),
    col('temperature'),
    col('feels_like'),
    col('humidity'),
    col('pressure'),
    col('wind_speed'),
    col('wind_direction'),
    col('rain_1h'),
    col('visibility'),
    col('weather_main'),
    col('weather_description'),
    col('year'),
    col('month'),
    col('day')
)

In [0]:
df_owm_clean.count()

10

In [0]:
df_owm_clean.limit(5).display()

city,country_code,join_time,temperature,feels_like,humidity,pressure,wind_speed,wind_direction,rain_1h,visibility,weather_main,weather_description,year,month,day
Bangkok,TH,2026-05-27T12:00:00.000Z,31.76,38.76,73,1006,5.46,192,0.0,10000,Clouds,broken clouds,2026,5,27
Berlin,DE,2026-05-27T12:00:00.000Z,22.52,21.74,35,1023,5.14,330,0.0,10000,Clear,clear sky,2026,5,27
Dubai,AE,2026-05-27T12:00:00.000Z,37.01,41.64,41,1005,5.14,270,0.0,10000,Clear,clear sky,2026,5,27
London,GB,2026-05-27T12:00:00.000Z,26.13,26.13,50,1026,3.13,74,0.0,10000,Clear,clear sky,2026,5,27
Mumbai,IN,2026-05-27T12:00:00.000Z,33.0,40.0,66,1008,5.66,260,0.0,7000,Haze,haze,2026,5,27


### Prepare TomTom for join (aggregate to city + hour)

In [0]:
df_tomtom_agg = df_tomtom \
    .groupBy('city', 'country_code', 'join_time') \
    .agg(
        spark_max('congestion_level').alias('congestion_level'),
        spark_count('icon_category').alias('incident_count')
    )


In [0]:
df_tomtom_agg.limit(5).display()

city,country_code,join_time,congestion_level,incident_count
Singapore,SG,2026-05-27T12:00:00.000Z,8,135
New York,US,2026-05-27T12:00:00.000Z,10,459
Bangkok,TH,2026-05-27T12:00:00.000Z,8,345
London,GB,2026-05-27T12:00:00.000Z,8,480
Dubai,AE,2026-05-27T12:00:00.000Z,8,66


In [0]:
df_tomtom_agg.count()

10

### Prepare OpenMeteo for join (select closest forecast hour)

In [0]:
df_meteo_clean = df_meteo.select(
    col('city'),
    col('country_code'),
    col('join_time'),
    col('temperature').alias('forecast_temperature'),
    col('rain').alias('forecast_rain'),
    col('wind_speed_10m').alias('forecast_wind_speed'),
    col('precipitation_probability')
).dropDuplicates(['city', 'join_time'])

In [0]:
df_meteo_clean.limit(5).display()

city,country_code,join_time,forecast_temperature,forecast_rain,forecast_wind_speed,precipitation_probability
Sydney,AU,2026-05-28T08:00:00.000Z,17.8,0.6,12.8,82
Sydney,AU,2026-05-28T11:00:00.000Z,17.4,0.4,15.7,85
Sydney,AU,2026-05-28T18:00:00.000Z,17.2,0.1,22.3,88
Sydney,AU,2026-05-30T04:00:00.000Z,18.2,0.0,13.9,0
Sydney,AU,2026-05-30T08:00:00.000Z,15.4,0.0,8.8,0


In [0]:
df_meteo_clean.count()

1680

In [0]:
df_fact = df_owm_clean \
    .join(
        df_tomtom_agg.select('city', 'join_time', 'congestion_level', 'incident_count'),
        ['city', 'join_time'], how='left'
    ) \
    .join(
        df_meteo_clean.select('city', 'join_time', 'forecast_temperature', 'forecast_rain',
                               'forecast_wind_speed', 'precipitation_probability'),
        ['city', 'join_time'], how='left'
    ) \
    .join(
        broadcast(dim_air_quality.select('city', 'dominant_pollutant', 'pm25', 'pm10', 'o3', 'no2')),
        ['city'], how='left'
    ) \
    .join(
        broadcast(dim_country.select('country_code', 'country_name', 'region', 'timezone', 'capital')),
        ['country_code'], how='left'
    ) \
    .fillna(0, subset=['congestion_level', 'incident_count', 'rain_1h', 'pm25', 'pm10'])

In [0]:
df_fact.select('city', 'country_code', 'join_time', 'temperature', 'congestion_level', 'pm25').limit(5).display()

city,country_code,join_time,temperature,congestion_level,pm25
Bangkok,TH,2026-05-27T12:00:00.000Z,31.76,8,39
Berlin,DE,2026-05-27T12:00:00.000Z,22.52,10,21
Dubai,AE,2026-05-27T12:00:00.000Z,37.01,8,95
London,GB,2026-05-27T12:00:00.000Z,26.13,8,65
Mumbai,IN,2026-05-27T12:00:00.000Z,33.0,8,151


In [0]:
df_fact.count()

10

### Add delay_flag and route_risk_score

In [0]:
df_fact = df_fact.withColumn('delay_flag',
    when(
        (col('rain_1h') > 5) |
        (col('congestion_level') > 7) |
        (col('pm25') > 150), 1
    ).otherwise(0)
)

In [0]:
df_fact = df_fact.withColumn('route_risk_score',
    round(
        # Weather component
        when(col('rain_1h') > 10, 30)
        .when(col('rain_1h') > 5,  20)
        .when(col('rain_1h') > 2,  10)
        .otherwise(0)
        +
        # Wind bonus
        when(col('wind_speed') > 15, 10)
        .when(col('wind_speed') > 8,  5)
        .otherwise(0)
        +
        # Traffic component
        when(col('congestion_level') > 8, 35)
        .when(col('congestion_level') > 6, 22)
        .when(col('congestion_level') > 3, 12)
        .otherwise(0)
        +
        # Air quality component
        when(col('pm25') > 150, 25)
        .when(col('pm25') > 100, 15)
        .when(col('pm25') > 50,   8)
        .otherwise(0)
    , 1)
)

In [0]:
df_fact.select('city','join_time','rain_1h','congestion_level','pm25','delay_flag','route_risk_score').limit(10).display()

city,join_time,rain_1h,congestion_level,pm25,delay_flag,route_risk_score
Bangkok,2026-05-27T12:00:00.000Z,0.0,8,39,1,22
Berlin,2026-05-27T12:00:00.000Z,0.0,10,21,1,35
Dubai,2026-05-27T12:00:00.000Z,0.0,8,95,1,30
London,2026-05-27T12:00:00.000Z,0.0,8,65,1,30
Mumbai,2026-05-27T12:00:00.000Z,0.0,8,151,1,47
New York,2026-05-27T12:00:00.000Z,0.0,10,30,1,35
Paris,2026-05-27T12:00:00.000Z,0.0,10,72,1,43
Sao Paulo,2026-05-27T12:00:00.000Z,0.0,8,109,1,37
Singapore,2026-05-27T12:00:00.000Z,0.0,8,68,1,30
Sydney,2026-05-27T12:00:00.000Z,0.84,10,43,1,35


### Final column selection (clean Gold schema)

In [0]:
df_gold = df_fact.select(
    # Keys
    col('city'),
    col('country_code'),
    col('join_time').alias('event_hour'),

    # Live weather (OWM)
    col('temperature'),
    col('feels_like'),
    col('humidity'),
    col('pressure'),
    col('wind_speed'),
    col('wind_direction'),
    col('rain_1h'),
    col('visibility'),
    col('weather_main'),
    col('weather_description'),

    # Forecast (OpenMeteo)
    col('forecast_temperature'),
    col('forecast_rain'),
    col('forecast_wind_speed'),
    col('precipitation_probability'),

    # Traffic (TomTom)
    col('congestion_level'),
    col('incident_count'),

    # Air Quality (WAQI)
    col('pm25'),
    col('pm10'),
    col('o3'),
    col('no2'),
    col('dominant_pollutant'),

    # Country dimension (REST Countries)
    col('country_name'),
    col('region'),
    col('timezone'),
    col('capital'),

    # KPIs
    col('delay_flag'),
    col('route_risk_score'),

    # Partitions
    col('year'),
    col('month'),
    col('day')
)

In [0]:
df_gold.printSchema()

root
 |-- city: string (nullable = true)
 |-- country_code: string (nullable = true)
 |-- event_hour: timestamp (nullable = true)
 |-- temperature: double (nullable = true)
 |-- feels_like: double (nullable = true)
 |-- humidity: long (nullable = true)
 |-- pressure: long (nullable = true)
 |-- wind_speed: double (nullable = true)
 |-- wind_direction: long (nullable = true)
 |-- rain_1h: double (nullable = false)
 |-- visibility: long (nullable = true)
 |-- weather_main: string (nullable = true)
 |-- weather_description: string (nullable = true)
 |-- forecast_temperature: double (nullable = true)
 |-- forecast_rain: double (nullable = true)
 |-- forecast_wind_speed: double (nullable = true)
 |-- precipitation_probability: long (nullable = true)
 |-- congestion_level: integer (nullable = false)
 |-- incident_count: long (nullable = false)
 |-- pm25: long (nullable = false)
 |-- pm10: long (nullable = false)
 |-- o3: double (nullable = true)
 |-- no2: double (nullable = true)
 |-- domina

In [0]:
print(f'Gold rows: {df_gold.count()}')
print(f'Gold columns: {len(df_gold.columns)}')

Gold rows: 10
Gold columns: 33


In [0]:
df_gold.select('city', 'country_code').distinct().orderBy('city').display()

city,country_code
Bangkok,TH
Berlin,DE
Dubai,AE
London,GB
Mumbai,IN
New York,US
Paris,FR
Sao Paulo,BR
Singapore,SG
Sydney,AU


In [0]:
df_gold.groupBy('delay_flag').count().display()

delay_flag,count
1,10


In [0]:
df_gold.groupBy('city') \
    .agg({'route_risk_score': 'avg', 'delay_flag': 'sum'}) \
    .withColumnRenamed('avg(route_risk_score)', 'avg_risk_score') \
    .withColumnRenamed('sum(delay_flag)', 'total_delays') \
    .orderBy('avg_risk_score', ascending=False) \
    .display()

city,avg_risk_score,total_delays
Mumbai,47.0,1
Paris,43.0,1
Sao Paulo,37.0,1
New York,35.0,1
Sydney,35.0,1
Berlin,35.0,1
Dubai,30.0,1
London,30.0,1
Singapore,30.0,1
Bangkok,22.0,1


### Write Gold Delta

In [0]:
gold_path = GOLD + 'logistics_gold/'

if DeltaTable.isDeltaTable(spark, gold_path):
    # Table exists — merge new rows in
    delta_table = DeltaTable.forPath(spark, gold_path)
    delta_table.alias('existing') \
        .merge(
            df_gold.alias('new'),
            'existing.city = new.city AND existing.event_hour = new.event_hour'
        ) \
        .whenMatchedUpdateAll() \
        .whenNotMatchedInsertAll() \
        .execute()
    print('Gold table merged successfully.')
else:
    # First run — create the table
    df_gold.write \
        .format('delta') \
        .mode('overwrite') \
        .option('overwriteSchema', 'true') \
        .partitionBy('year', 'month', 'day') \
        .save(gold_path)
    print('Gold table created successfully.')

Gold table created successfully.
